# Paper — Fig 1 + Fig 2: Orientation Artifact Analysis

**Produces:**
- `figures_paper/fig1_histograms.pdf` — orientation histograms for GT + all 4 models
- `figures_paper/fig2_synthetic.pdf` — synthetic rasterization experiment (2 panels)
- KS statistics printed to stdout

**No GPU required.**

Key design choice: area threshold applied to GT only, NOT predictions.
SAM2 tight masks would be unfairly cut by the same threshold used for GT labeling.

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Polygon
from shapely import segmentize
from scipy.stats import kstest
from tqdm import tqdm

from rastertools_BOULDERING import metadata as raster_metadata
from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

In [ ]:
prieur_dir      = Path("/scratch/users/cayleigh/Apr2023-Mars-Moon-Earth-mask-5px")
prieur_test_dir = prieur_dir / "preprocessing" / "test"
work_dir        = Path.home() / "tmp" / "YOLOv8BeyondEarth"
in_raster       = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")

gt_tile_ids = ["1386", "1503", "2054", "2277", "2508"]

PRED_CONFIGS = {
    "YOLOv8":          (work_dir / "exp_yolo_256",           "*-downscaled-mask-nms.shp"),
    "SAM2 zero-shot":  (work_dir / "exp_sam2_256",           "*-downscaled-mask-nms.shp"),
    "SAM2 fine-tuned": (work_dir / "exp_sam2_finetuned_256", "*-downscaled-mask-nms.shp"),
    "SAM2-auto":       (work_dir / "exp_sam2_auto_256",      "*-mask-nms.shp"),
}

res             = raster_metadata.get_resolution(in_raster)[0]
AREAL_THRESHOLD = (res ** 2) * (4.74 ** 2)   # applied to GT only
AR_MIN, AR_MAX  = 1.2, 2.0
BINS            = np.linspace(0, 180, 37)     # 5° bins

OUT_DIR = Path("figures_paper"); OUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 9, "axes.titlesize": 9, "figure.dpi": 150})
print(f"Resolution: {res:.4f} m/px   GT areal threshold: {AREAL_THRESHOLD:.4f} m²")

In [ ]:
def run_pipeline(poly, res):
    """segmentize → fitEllipse → MRR → (angle180, aspect_ratio) or None."""
    row_seg = pd.Series({"geometry": segmentize(poly, res)})
    try:
        ellipse_poly, _, _, _ = fitEllipse(row_seg)
    except Exception:
        return None
    try:
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        _, _, long_ax, short_ax, _, _, _, angle180 = boulder_row(mrr_row)
    except Exception:
        return None
    if short_ax < 1e-6:
        return None
    return angle180, long_ax / short_ax

In [ ]:
print("Loading GT polygons...")
gt_shp_paths = [
    prieur_test_dir / "labels" / f"M1221383405_{tid}_mask.shp"
    for tid in gt_tile_ids
    if (prieur_test_dir / "labels" / f"M1221383405_{tid}_mask.shp").exists()
]
print(f"  Found {len(gt_shp_paths)}/{len(gt_tile_ids)} GT shapefiles")

gdfs = [gpd.read_file(p) for p in gt_shp_paths]
gdf_gt = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
gdf_gt["poly_area"] = gdf_gt.geometry.area
gdf_gt = gdf_gt[gdf_gt["poly_area"] >= AREAL_THRESHOLD].reset_index(drop=True)
print(f"  GT after area filter: {len(gdf_gt):,}")

gt_angles = []
for geom in tqdm(gdf_gt.geometry, desc="GT orientation", leave=False):
    if geom is None or geom.is_empty:
        continue
    r = run_pipeline(geom, res)
    if r and AR_MIN <= r[1] <= AR_MAX:
        gt_angles.append(r[0])
gt_angles = np.array(gt_angles)
print(f"  GT elongated boulders (AR {AR_MIN}–{AR_MAX}): {len(gt_angles)}")

In [ ]:
pred_angles = {}

for name, (pred_dir, glob) in PRED_CONFIGS.items():
    shp_paths = sorted(pred_dir.glob(glob))
    if not shp_paths:
        print(f"  [{name}] No shapefiles in {pred_dir} — skipping")
        continue

    gdfs = [gpd.read_file(p) for p in shp_paths]
    gdf  = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    gdf["poly_area"] = gdf.geometry.area
    n_total = len(gdf)
    # No area filter on predictions — SAM2 tight masks are smaller than YOLO
    # but still valid. Area filter applies to GT only.

    angles = []
    for geom in tqdm(gdf.geometry, desc=name, leave=False):
        if geom is None or geom.is_empty:
            continue
        r = run_pipeline(geom, res)
        if r and AR_MIN <= r[1] <= AR_MAX:
            angles.append(r[0])
    n_elongated = len(angles)

    print(f"{name}:")
    print(f"  total detections: {n_total:>7,}")
    print(f"  elongated (AR {AR_MIN}–{AR_MAX}): {n_elongated:>7,}  ({n_elongated/n_total*100:.1f}%)")
    print()
    pred_angles[name] = np.array(angles)

## Figure 1 — Orientation histograms

GT should be flat; all model variants should show orientation spikes.

In [ ]:
def hist_ax(angles, ax, title, color, gt_angles=None):
    counts, _ = np.histogram(angles, bins=BINS)
    cx = (BINS[:-1] + BINS[1:]) / 2
    ax.bar(cx, counts, width=4.5, color=color, edgecolor="white", lw=0.3)
    if gt_angles is not None:
        gc, _ = np.histogram(gt_angles, bins=BINS)
        ax.step(cx, gc * len(angles) / max(len(gt_angles), 1),
                where="mid", color="seagreen", lw=1.2, label="GT (scaled)")
        ax.legend(fontsize=7)
    ax.axhline(len(angles) / len(cx), color="k", ls="--", lw=0.6, alpha=0.5)
    ax.set(xlim=(0, 180), xticks=[0, 45, 90, 135, 180],
           xlabel="Orientation (°)", ylabel="Count")
    ax.set_title(f"{title}\n(n={len(angles):,})")
    ax.spines[["top", "right"]].set_visible(False)

all_series = [("GT (Prieur et al.)", gt_angles, "seagreen")] + \
             [(n, a, "#4C72B0") for n, a in pred_angles.items()]

fig, axes = plt.subplots(1, len(all_series), figsize=(2.8 * len(all_series), 2.8))
if len(all_series) == 1:
    axes = [axes]
for ax, (name, angles, color) in zip(axes, all_series):
    hist_ax(angles, ax, name, color)
axes[0].set_facecolor("#f5fbf5")

fig.tight_layout()
fig.savefig(OUT_DIR / "fig1_histograms.pdf", bbox_inches="tight")
plt.show()
print("Saved fig1_histograms.pdf")

## KS statistics vs. uniform

In [ ]:
print(f"KS test vs. uniform (D statistic, p-value):")
print(f"{'Method':<24}  {'n':>7}  {'D':>6}  {'p':>10}")
print("-" * 52)
for name, angles, _ in all_series:
    D, p = kstest(angles / 180.0, "uniform")
    print(f"{name:<24}  {len(angles):>7,}  {D:>6.3f}  {p:>10.2e}")

## Figure 2 — Synthetic rasterization experiment

Smooth ellipses at known angles → rasterize to pixel-aligned mask → run pipeline.
Expected: smooth recovers correct angle; pixel-aligned snaps to cardinal orientations.

In [ ]:
def make_smooth_ellipse(a, b, theta_deg, cx=0.0, cy=0.0, n_pts=200):
    theta = np.radians(theta_deg)
    t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    x = a * np.cos(t) * np.cos(theta) - b * np.sin(t) * np.sin(theta) + cx
    y = a * np.cos(t) * np.sin(theta) + b * np.sin(t) * np.cos(theta) + cy
    return Polygon(np.column_stack([x, y]))


def rasterize_to_pixel_aligned(poly, pix_per_unit=10):
    minx, miny, maxx, maxy = poly.bounds
    w   = int((maxx - minx) * pix_per_unit) + 20
    h   = int((maxy - miny) * pix_per_unit) + 20
    pad = 5
    pts    = np.array(poly.exterior.coords[:-1])
    pts_px = ((pts - [minx, miny]) * pix_per_unit + pad).astype(np.int32)
    canvas = np.zeros((h, w), np.uint8)
    cv2.fillPoly(canvas, [pts_px], 255)
    cnts, _ = cv2.findContours(canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    cnt = max(cnts, key=cv2.contourArea).squeeze()
    if cnt.ndim != 2 or len(cnt) < 4:
        return None
    coords = (cnt.astype(float) - pad) / pix_per_unit + [minx, miny]
    return Polygon(coords)


A, B      = 15.0, 10.0
RES_SYNTH = 0.5
N         = 300

np.random.seed(42)
input_angles = np.random.uniform(0, 180, N)

smooth_recovered, pixalign_recovered = [], []
for theta in tqdm(input_angles, desc="Synthetic test"):
    smooth_poly = make_smooth_ellipse(A, B, theta)
    px_poly     = rasterize_to_pixel_aligned(smooth_poly)

    r_s = run_pipeline(smooth_poly, RES_SYNTH)
    smooth_recovered.append(r_s[0] if (r_s and AR_MIN <= r_s[1] <= AR_MAX) else None)

    if px_poly is not None:
        r_p = run_pipeline(px_poly, RES_SYNTH)
        pixalign_recovered.append(r_p[0] if (r_p and AR_MIN <= r_p[1] <= AR_MAX) else None)
    else:
        pixalign_recovered.append(None)

valid = [(s, p) for s, p in zip(smooth_recovered, pixalign_recovered)
         if s is not None and p is not None]
sm_a  = np.array([v[0] for v in valid])
px_a  = np.array([v[1] for v in valid])
print(f"Valid: {len(valid)}/{N}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(5.6, 2.8))

hist_ax(sm_a, axes[0], "Smooth polygon\n(correct)",                     "seagreen")
hist_ax(px_a, axes[1], "Pixel-aligned polygon\n(after rasterization)",  "firebrick")

fig.tight_layout()
fig.savefig(OUT_DIR / "fig2_synthetic.pdf", bbox_inches="tight")
plt.show()
print("Saved fig2_synthetic.pdf")